# Cat Identity Fine-tuning

Fine-tunes YOLOv8m on the `salo`/`taro` dataset.

**Before running:**
1. Run `scripts/split_dataset.py` locally to prepare the dataset
2. Zip `data/labeled/` — `cd /path/to/catpoint-cv && zip -r labeled.zip data/labeled/`
3. Upload `labeled.zip` to your Google Drive
4. Set `DRIVE_ZIP_PATH` below to the path in your Drive

**After running:**
1. Download `best.pt` from the output path shown at the end
2. Run `python scripts/export_openvino.py --model best.pt` locally

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
DRIVE_ZIP_PATH = "MyDrive/catpoint-cv/labeled.zip"  # path in your Google Drive
EPOCHS         = 100
BATCH          = 16   # T4 can handle 16 at imgsz=1280
IMGSZ          = 1280
MODEL          = "yolov8m.pt"
RUN_NAME       = "cat_identity"

In [ ]:
# ── Verify GPU ────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU found — change runtime to GPU")

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
%pip install ultralytics pyyaml -q

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Extract dataset ───────────────────────────────────────────────────────────
import zipfile
from pathlib import Path

zip_path = Path("/content/drive") / DRIVE_ZIP_PATH
assert zip_path.exists(), f"ZIP not found: {zip_path}"

with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")

dataset_yaml = Path("/content/data/labeled/dataset.yaml")
assert dataset_yaml.exists(), "dataset.yaml not found after extraction"
print(f"Dataset ready: {dataset_yaml}")
print(dataset_yaml.read_text())

In [ ]:
# ── Patch dataset.yaml path for Colab ────────────────────────────────────────
# The path field in dataset.yaml points to your local machine.
# We need to update it to the Colab path.
import yaml

with open(dataset_yaml) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/content/data/labeled"

with open(dataset_yaml, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Patched dataset.yaml:")
print(dataset_yaml.read_text())

In [ ]:
# ── Fine-tune ─────────────────────────────────────────────────────────────────
from ultralytics import YOLO

BACKBONE_FREEZE_LAYERS = 10

model = YOLO(MODEL)

# Freeze backbone
freeze = [f"model.{i}." for i in range(BACKBONE_FREEZE_LAYERS)]
for name, param in model.model.named_parameters():
    if any(name.startswith(f) for f in freeze):
        param.requires_grad = False
frozen = sum(1 for _, p in model.model.named_parameters() if not p.requires_grad)
total  = sum(1 for _ in model.model.named_parameters())
print(f"Frozen {frozen}/{total} parameter tensors")

results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    name=RUN_NAME,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.3,
    flipud=0.1,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    dropout=0.1,
    weight_decay=0.0005,
    plots=True,
    save=True,
    save_period=10,
)

best = results.save_dir + "/weights/best.pt"
print(f"\nBest checkpoint: {best}")

In [ ]:
# ── Save best checkpoint to Drive ────────────────────────────────────────────
import shutil

output_dir = Path("/content/drive/MyDrive/catpoint-cv/checkpoints")
output_dir.mkdir(parents=True, exist_ok=True)

dest = output_dir / "best.pt"
shutil.copy2(best, dest)
print(f"Saved to Drive: {dest}")
print("Download this file and run:")
print("  python scripts/export_openvino.py --model best.pt")